# Zoom 녹화 → 텍스트 변환 (Whisper)
2시간 회의 녹화본을 텍스트로 변환합니다.

In [ ]:
# 1. GPU 확인 & Whisper 설치
!nvidia-smi
!pip install -q openai-whisper

In [ ]:
# 2. Google Drive 마운트
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 3. 파일 경로 설정
# ★ 여기에 Drive 내 파일 경로를 넣으세요
# 예: /content/drive/MyDrive/회의녹화.mp4
import os

# Drive 최상위에서 영상 파일 찾기
drive_root = '/content/drive/MyDrive'
video_files = [f for f in os.listdir(drive_root) if f.endswith(('.mp4','.mkv','.webm','.m4a','.wav','.mp3'))]
print('찾은 영상/오디오 파일:')
for i, f in enumerate(video_files):
    size_mb = os.path.getsize(os.path.join(drive_root, f)) / 1024 / 1024
    print(f'  [{i}] {f} ({size_mb:.0f}MB)')

In [ ]:
# 4. 파일 선택 (위 목록에서 번호 입력하거나 직접 경로 지정)
# 방법1: 번호로 선택
FILE_INDEX = 0  # ← 위 목록에서 해당 번호로 변경
INPUT_FILE = os.path.join(drive_root, video_files[FILE_INDEX])

# 방법2: 직접 경로 지정 (방법1 대신 이걸 쓰려면 아래 주석 해제)
# INPUT_FILE = '/content/drive/MyDrive/폴더명/파일명.mp4'

print(f'선택된 파일: {INPUT_FILE}')
print(f'파일 크기: {os.path.getsize(INPUT_FILE)/1024/1024:.0f}MB')

In [ ]:
# 5. 오디오 추출 (영상→오디오, 이미 오디오면 스킵)
AUDIO_FILE = '/content/audio.wav'

if INPUT_FILE.endswith(('.wav','.mp3','.m4a')):
    AUDIO_FILE = INPUT_FILE
    print('이미 오디오 파일이므로 추출 생략')
else:
    print('오디오 추출 중...')
    !ffmpeg -y -i "{INPUT_FILE}" -vn -acodec pcm_s16le -ar 16000 -ac 1 "{AUDIO_FILE}" -loglevel quiet
    size = os.path.getsize(AUDIO_FILE) / 1024 / 1024
    print(f'추출 완료: {size:.0f}MB')

In [ ]:
# 6. Whisper 실행 (medium 모델, 한국어)
import whisper
import time

print('모델 로딩 중 (medium)...')
model = whisper.load_model('medium')

print('텍스트 변환 시작... (20~30분 소요)')
start = time.time()

result = model.transcribe(
    AUDIO_FILE,
    language='ko',
    verbose=True  # 진행상황 표시
)

elapsed = time.time() - start
print(f'\n완료! 소요시간: {elapsed/60:.1f}분')

In [ ]:
# 7. 결과 저장 (타임스탬프 포함)
def format_time(seconds):
    h = int(seconds // 3600)
    m = int((seconds % 3600) // 60)
    s = int(seconds % 60)
    return f'{h:02d}:{m:02d}:{s:02d}'

# 타임스탬프 버전
output_ts = os.path.join(drive_root, '회의록_타임스탬프.txt')
with open(output_ts, 'w', encoding='utf-8') as f:
    f.write('=== 회의 녹취록 (타임스탬프) ===\n\n')
    for seg in result['segments']:
        start_t = format_time(seg['start'])
        end_t = format_time(seg['end'])
        f.write(f'[{start_t} ~ {end_t}] {seg["text"].strip()}\n')

# 전체 텍스트 버전
output_full = os.path.join(drive_root, '회의록_전체.txt')
with open(output_full, 'w', encoding='utf-8') as f:
    f.write('=== 회의 녹취록 ===\n\n')
    f.write(result['text'])

print(f'저장 완료!')
print(f'  타임스탬프: {output_ts}')
print(f'  전체 텍스트: {output_full}')
print(f'\n총 세그먼트: {len(result["segments"])}개')
print(f'전체 텍스트 길이: {len(result["text"]):,}자')

In [ ]:
# 8. 미리보기 (처음 20개 세그먼트)
print('=== 미리보기 (처음 20개) ===\n')
for seg in result['segments'][:20]:
    start_t = format_time(seg['start'])
    print(f'[{start_t}] {seg["text"].strip()}')